# SRE Environment — GRPO Training Notebook

Kaggle-ready GRPO training notebook for a Llama-based SRE agent on **2× T4 GPUs**.
The repo is cloned from GitHub so this notebook is fully self-contained — no local
checkout required.

**Training loop:**
1. The model receives a one-line pager alert and the current broken-server terminal output
2. It generates **exactly one shell command**
3. The SRE environment executes the command and returns real terminal output + a dense reward
4. GRPO updates weights using relative advantage over `num_generations` rollouts per prompt

**Reward signal:** +2.5 – +3.5 for a verified system fix; −0.02 step cost per command;
−2.0 for destructive commands (episode-ending). No reward hacking — reward comes from
live system state, not model text.

---

## Setup — Clone Repository and Start SRE Server

Clones `SRE-Env` from GitHub, installs **nginx** (needed for fault simulation in
HF/subprocess mode), sets `SANDBOX_MODE=hf` (no Docker required on Kaggle), and starts
the FastAPI SRE server as a background process on **port 8000**.

> **Kaggle note:** Some health checks call `systemctl`. On Kaggle (no systemd), those
> calls return non-zero and are skipped gracefully — health scoring still works for all
> process-based and file-based checks.

In [ ]:
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/Its-Atharva-Gupta/SRE-Env.git'
REPO_DIR = '/kaggle/working/SRE-Env'

# ── clone or update repo ────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed:\n{r.stderr}')
    print('Cloned.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)
    print('Repo up to date.')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# ── install nginx for fault simulation ──────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', '-qq', 'nginx'], check=False)
subprocess.run(['service', 'nginx', 'start'], check=False)
print('nginx started.')

# ── subprocess-based sandbox (no Docker on Kaggle) ──────────────────────────
os.environ['SANDBOX_MODE'] = 'hf'
os.environ['PYTHONPATH']   = REPO_DIR

print(f'Working dir  : {os.getcwd()}')
print(f'SANDBOX_MODE : {os.environ["SANDBOX_MODE"]}')

## Cell 1 — Install Dependencies

Installs all packages needed for training using `uv` (fast resolver) with pinned versions
for reproducibility on Kaggle's T4 runtime.

**Install logic:**
- If `torch` is absent **or** the runtime is Colab/Kaggle (detected via `COLAB_` env vars),
  do a full cold install: PyTorch ≥ 2.8, Triton ≥ 3.4 (pinned commit for T4 stability),
  `transformers==4.56.2`, `unsloth` and `unsloth_zoo` from their latest GitHub HEAD
- If `torch` is present but `unsloth` is missing (e.g. a base Python environment),
  install only `unsloth` + `trackio`
- Either way, a final `--no-deps` upgrade pins `transformers==4.56.2`, `trl==0.22.2`,
  and refreshes `unsloth` / `unsloth_zoo` without pulling in conflicting transitive deps

`pip install -e .` installs the `sre_env` package itself so that `from client import SREEnv` works.

In [ ]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo
!pip install -e . -q

## Start SRE Server

Launches the FastAPI SRE environment server as a **background subprocess** on port 8000.
The server handles episode resets and step execution via the OpenEnv REST protocol.

Must run **before** any `env.reset()` or `env.step()` calls.
The cell polls `/health` and blocks until the server is ready (up to 30 s).

In [ ]:
import requests

server_log  = open('/tmp/sre_server.log', 'w')
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd=REPO_DIR,
    stdout=server_log,
    stderr=server_log,
    env={**os.environ, 'SANDBOX_MODE': 'hf', 'PYTHONPATH': REPO_DIR},
)

print('Waiting for SRE server', end='', flush=True)
for _ in range(30):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(' ready.')
            break
    except Exception:
        time.sleep(1)
        print('.', end='', flush=True)
else:
    print()
    print('[WARN] Server may not be ready — check /tmp/sre_server.log')

## Cell 2 — Imports

Standard imports for the training loop. `SREEnv` and `SREAction` are the OpenEnv client
wrappers for the SRE environment. A fallback import handles both the installed-package
path and the direct-from-repo path.

In [ ]:
import os, re, json, textwrap
from datetime import datetime
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

try:
    from sre_env.client import SREEnv, SREAction   # installed package
except ImportError:
    from client import SREEnv, SREAction            # direct from repo

## Cell 3 — Configuration

Single dict that controls every training hyperparameter and toggle.
**Change values here only** — all downstream cells read from `CFG`.

Key settings for 2× T4 (16 GB each):
- `batch_size=2` + `grad_accum=4` → effective batch of 8
- `num_generations=8` → 8 rollouts per prompt for GRPO advantage estimation
- `max_new_tokens=128` → one-command responses are short; cap prevents runaway generation
- `load_in_4bit=False` in Cell 6 — Llama-3.2-3B fits in bf16 across 2× T4

In [ ]:
CFG = {
    'openenv_url':       'http://localhost:8000',
    'model_name':        'meta-llama/Llama-3.2-3B-Instruct',
    'max_seq_length':    2048,
    'lora_r':            16,
    'lora_alpha':        16,
    'dataset_size':      500,
    'num_generations':   8,
    'batch_size':        2,
    'grad_accum':        4,
    'epochs':            3,
    'lr':                5e-6,
    'warmup_steps':      20,
    'max_new_tokens':    128,
    'output_dir':        'outputs/grpo',
    'log_every_n_steps': 1,
    'wandb_project':     'sre-env',
    'use_wandb':         False,   # flip to True if wandb is available
}

## Cell 4 — WandB Init (Optional)

Initialises a Weights & Biases run when `CFG["use_wandb"]` is `True`.
On Kaggle, add your API key as a secret named `WANDB_API_KEY` and set
`use_wandb=True` in CFG before running this cell.

In [ ]:
if CFG['use_wandb']:
    import wandb
    wandb.init(project=CFG['wandb_project'], config=CFG)

## Cell 5 — Episode Logger

`EpisodeLogger` is the most important observability tool in this notebook.
It logs and pretty-prints **every** interaction: model output, extracted command,
raw terminal response, and a per-component reward breakdown for every training rollout.

- `begin_episode` / `end_episode` bracket each rollout with a visual header and footer
- `log_step` prints the full step breakdown including health score and passing stages
- `save_log` dumps all episodes to JSON for post-training analysis and reward debugging
- WandB metrics are emitted per-step and per-episode when enabled

The logger instance `logger` is module-level so `reward_fn` (Cell 10) can write to it.

In [ ]:
class EpisodeLogger:
    def __init__(self):
        self.episodes     = []
        self.current      = None
        self.step_count   = 0
        self.episode_count = 0

    def begin_episode(self, fault_id: str, initial_obs: str, prompt: str):
        self.episode_count += 1
        self.current = {
            'episode_id':   self.episode_count,
            'fault_id':     fault_id,
            'initial_obs':  initial_obs,
            'prompt':       prompt,
            'steps':        [],
            'total_reward': 0.0,
            'success':      False,
            'timestamp':    datetime.now().isoformat(),
        }
        self._print_episode_header()

    def log_step(self, step: int, command: str, raw_output: str,
                 terminal_output: str, reward: float, done: bool, info: dict):
        self.step_count += 1
        self.current['total_reward'] += reward
        self.current['success'] = info.get('verified', False)
        step_data = {
            'step':             step,
            'command':          command,
            'raw_model_output': raw_output,
            'terminal_output':  terminal_output,
            'reward':           reward,
            'done':             done,
            'health_score':     info.get('health_score', 0.0),
            'passing_stages':   info.get('passing_stages', []),
            'r_progress':       info.get('r_progress', 0.0),
            'r_action':         info.get('r_action', 0.0),
            'r_terminal':       info.get('r_terminal', 0.0),
            'r_penalty':        info.get('r_penalty', 0.0),
        }
        self.current['steps'].append(step_data)
        self._print_step(step_data)

    def end_episode(self):
        self.episodes.append(self.current)
        self._print_episode_footer()
        if CFG['use_wandb']:
            import wandb
            wandb.log({
                'episode/total_reward': self.current['total_reward'],
                'episode/success':      int(self.current['success']),
                'episode/steps_taken':  len(self.current['steps']),
                'episode/fault_id':     self.current['fault_id'],
                'episode/count':        self.episode_count,
            })
        self.current = None

    def _print_episode_header(self):
        print('\n' + '█' * 70)
        print(f"  EPISODE {self.current['episode_id']:04d}  |  fault: {self.current['fault_id']}")
        print('█' * 70)
        print('\n── INITIAL OBSERVATION ' + '─' * 49)
        print(self.current['initial_obs'])
        print('\n── PROMPT SENT TO MODEL ' + '─' * 48)
        print(self.current['prompt'])
        print('─' * 70)

    def _print_step(self, s: dict):
        print(f"\n┌── STEP {s['step']} " + '─' * 61)
        print(f"│  RAW MODEL OUTPUT : {repr(s['raw_model_output'])}")
        print(f"│  EXTRACTED COMMAND: {s['command']}")
        print(f"│  TERMINAL OUTPUT  :")
        for line in s['terminal_output'].splitlines():
            print(f'│    {line}')
        print(f"│  REWARD BREAKDOWN :")
        print(f"│    progress  = {s['r_progress']:+.4f}")
        print(f"│    action    = {s['r_action']:+.4f}")
        print(f"│    terminal  = {s['r_terminal']:+.4f}")
        print(f"│    penalty   = {s['r_penalty']:+.4f}")
        print(f"│    " + '─' * 17)
        print(f"│    TOTAL     = {s['reward']:+.4f}")
        print(f"│  HEALTH SCORE     : {s['health_score']:.3f}")
        print(f"│  PASSING STAGES   : {s['passing_stages']}")
        print(f"│  DONE             : {s['done']}")
        print('└' + '─' * 65)

    def _print_episode_footer(self):
        e = self.current
        status = '✓ SUCCESS' if e['success'] else '✗ FAILED'
        print('\n' + '═' * 70)
        print(f"  {status}  |  total reward: {e['total_reward']:+.4f}  |  steps: {len(e['steps'])}")
        print('═' * 70 + '\n')

    def save_log(self, path: str = 'outputs/episode_log.json'):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w') as f:
            json.dump(self.episodes, f, indent=2)
        print(f'[Logger] Saved {len(self.episodes)} episodes to {path}')

logger = EpisodeLogger()

## Cell 6 — Load Model and Tokenizer via Unsloth

Loads `Llama-3.2-3B-Instruct` in bf16 using Unsloth's `FastLanguageModel`, then wraps
it with a LoRA adapter for efficient fine-tuning.

- `load_in_4bit=False` — Llama-3.2-3B in bf16 fits across 2× T4 (≈ 6 GB per GPU)
- `use_gradient_checkpointing="unsloth"` — Unsloth's optimised checkpointing cuts
  activation memory by ~30 % vs standard checkpointing
- All projection layers are targeted (q, k, v, o, gate, up, down) for full LoRA coverage

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG['model_name'],
    max_seq_length=CFG['max_seq_length'],
    dtype=None,
    load_in_4bit=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG['lora_r'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f"Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}")

## Cell 7 — Connect to Environment

Creates an `SREEnv` client pointed at the local server started in the Setup cell.
Runs a single `reset()` call to confirm the connection is healthy and prints
the initial observation so you can verify the server is injecting faults correctly.

In [ ]:
env = SREEnv(base_url=CFG['openenv_url'])
print('Connected to SRE environment.')
obs = env.reset()
print('Test reset successful. Initial observation:')
print(obs.terminal_output)

## Cell 8 — System Prompt and Prompt Builder

`SYSTEM_PROMPT` instructs the model to behave as a strict SRE: respond with exactly
one shell command and nothing else. Verbosity and explanations are penalised implicitly
because they waste the 128-token budget on non-commands.

`build_prompt` formats the current observation into a chat template using the tokenizer's
`apply_chat_template` so the model sees the correct special tokens for Llama-3.

`extract_command` strips markdown formatting, backticks, and whitespace from the model's
raw output to get a clean executable command.

In [ ]:
SYSTEM_PROMPT = (
    'You are an SRE (Site Reliability Engineer). '
    'You have SSH access to a production Linux server that has an active incident. '
    'You will receive an alert and the current terminal state. '
    'Respond with exactly ONE shell command to run. '
    'No explanation. No markdown. No backticks. No comments. '
    'Just the raw shell command and nothing else.'
)

def build_prompt(obs) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'{obs.alert}\n\n{obs.terminal_output}\n\n[sre@prod-01 ~]$'},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def extract_command(raw_output: str) -> str:
    """Pull the first non-empty line; strip backticks and surrounding whitespace."""
    for line in raw_output.strip().splitlines():
        line = line.strip().strip('`').strip()
        if line:
            return line
    return raw_output.strip()

## Cell 9 — Build Prompt Dataset

Generates `CFG["dataset_size"]` prompts by repeatedly calling `env.reset()`.
Each reset samples a random fault from the registry, injects it, and returns
the initial observation (pager alert + diagnostic output).

The prompts are stored as a HuggingFace `Dataset` — this is what `GRPOTrainer`
iterates over during training. Each training step will generate `num_generations`
rollouts from the same prompt and compute relative advantages.

In [ ]:
def build_dataset(n: int = CFG['dataset_size']) -> Dataset:
    prompts = []
    print(f'Generating {n} prompts from environment resets...')
    for i in range(n):
        obs    = env.reset()
        prompt = build_prompt(obs)
        prompts.append({'prompt': prompt})
        if (i + 1) % 50 == 0:
            print(f'  {i + 1}/{n}')
    print(f'Dataset built: {len(prompts)} prompts')
    return Dataset.from_list(prompts)

dataset = build_dataset()

## Cell 10 — Reward Function with Full Logging

`reward_fn` is the bridge between the model's generated commands and the SRE environment.
It is called by `GRPOTrainer` once per training step with a flat list of completions
(all `num_generations` rollouts for the current batch, interleaved).

**Per-completion logic:**
1. Extract the shell command from raw model output
2. Call `env.reset()` for a fresh broken-server state (each rollout is independent)
3. Call `env.step(command)` to execute the command and receive the dense reward
4. Log everything via `EpisodeLogger` — every command, output, and reward breakdown
5. Emit per-step WandB metrics if enabled

Errors (network timeouts, empty commands) return `−1.0` so training continues gracefully.

In [ ]:
def reward_fn(completions, prompts=None, **kwargs) -> list[float]:
    rewards = []
    for i, raw_output in enumerate(completions):
        command = extract_command(raw_output)
        try:
            # Fresh episode for each rollout so rollouts are independent
            obs = env.reset()
            logger.begin_episode(
                fault_id=obs.alert[:40],
                initial_obs=obs.terminal_output,
                prompt=build_prompt(obs),
            )

            obs, reward, done, info = env.step(SREAction(command=command))

            logger.log_step(
                step=1,
                command=command,
                raw_output=raw_output,
                terminal_output=obs.terminal_output,
                reward=reward,
                done=done,
                info=info,
            )
            logger.end_episode()
            rewards.append(float(reward))

            if CFG['use_wandb']:
                import wandb
                wandb.log({
                    'step/reward':       reward,
                    'step/health_score': info.get('health_score', 0.0),
                    'step/r_progress':   info.get('r_progress', 0.0),
                    'step/r_action':     info.get('r_action', 0.0),
                    'step/r_terminal':   info.get('r_terminal', 0.0),
                    'step/r_penalty':    info.get('r_penalty', 0.0),
                    'step/command':      command,
                })

        except Exception as e:
            print(f'  [reward_fn] Error on completion {i}: {e}')
            if logger.current:
                logger.end_episode()
            rewards.append(-1.0)

    return rewards

## Cell 11 — GRPOConfig and Trainer

Configures `GRPOTrainer` with the hyperparameters from `CFG`.

Key GRPO settings:
- `num_generations=8` — 8 rollouts per prompt; GRPO computes relative advantage
  across these 8 rewards to create the policy gradient signal
- `max_prompt_length=1024` — truncates overly long prompts to fit within sequence budget
- `bf16=True` — bf16 is more numerically stable than fp16 for LoRA training on T4s
- `remove_unused_columns=False` — required because the dataset has only a `prompt` column
  and GRPO needs to keep it intact

In [ ]:
config = GRPOConfig(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    num_generations=CFG['num_generations'],
    max_new_tokens=CFG['max_new_tokens'],
    max_prompt_length=1024,
    bf16=True,
    fp16=False,
    learning_rate=CFG['lr'],
    warmup_steps=CFG['warmup_steps'],
    logging_steps=CFG['log_every_n_steps'],
    save_steps=50,
    report_to='wandb' if CFG['use_wandb'] else 'none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=dataset,
)

## Cell 12 — Train

Starts the GRPO training loop. Each step:
1. Samples a batch of prompts from the dataset
2. Generates `num_generations` completions per prompt using the current policy
3. Calls `reward_fn` to evaluate all completions (executing commands in the SRE env)
4. Computes group-relative advantages: `A_i = (r_i − mean(r)) / std(r)`
5. Updates the model with the clipped policy gradient loss

All episode interactions are printed live by `EpisodeLogger`. Look for the `TOTAL` reward
line per step — it should trend upward over training as the model learns to fix servers
rather than spam `ls`.

In [ ]:
print('Starting GRPO training...')
print(f'  Model       : {CFG["model_name"]}')
print(f'  Dataset     : {len(dataset)} prompts')
print(f'  Generations : {CFG["num_generations"]} rollouts per prompt')
print(f'  Epochs      : {CFG["epochs"]}')
print(f'  GPUs        : {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}')
print()

trainer.train()

## Cell 13 — Save Checkpoint

Saves three artefacts:

| Path | Contents |
|---|---|
| `outputs/sre-lora/` | LoRA adapter weights (small — adapter only) |
| `outputs/sre-merged/` | Full merged 16-bit model (ready for inference) |
| `outputs/episode_log.json` | Complete episode log for reward analysis |

On Kaggle, save to `/kaggle/working/` (already the working dir) and the outputs folder
will be available as a dataset output after the run completes.

In [ ]:
model.save_pretrained('outputs/sre-lora')
tokenizer.save_pretrained('outputs/sre-lora')
model.save_pretrained_merged('outputs/sre-merged', tokenizer, save_method='merged_16bit')
logger.save_log('outputs/episode_log.json')
print('Training complete. Model and logs saved.')

## Cell 14 — Quick Eval (Run After Training)

Runs **10 episodes** with the trained model using **greedy decoding** (`do_sample=False`).
Each episode loops up to 8 steps: generate command → execute → observe → repeat.

The same `EpisodeLogger` format is used so every command, terminal output, and reward
breakdown is printed in full. At the end, prints:

- **Success rate** — fraction of episodes where the fault was verified as fixed
- **Mean reward** — average total episode reward (compare to random-policy baseline −0.1 to +0.3)
- **Per-episode reward list** — quick scan for outliers

A trained model solving `broken_nginx_config` should score +2.5 to +3.5 in ≤ 4 steps.
A random policy typically scores −0.5 to +0.3.

In [ ]:
success_count = 0
eval_rewards  = []
eval_logger   = EpisodeLogger()

FastLanguageModel.for_inference(model)   # re-enable Unsloth inference optimisations

for ep in range(10):
    obs    = env.reset()
    prompt = build_prompt(obs)
    eval_logger.begin_episode(obs.alert[:40], obs.terminal_output, prompt)
    total_reward = 0.0

    for step in range(8):
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
            )
        raw     = tokenizer.decode(
            output[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        )
        command = extract_command(raw)
        obs, reward, done, info = env.step(SREAction(command=command))

        eval_logger.log_step(step + 1, command, raw, obs.terminal_output, reward, done, info)
        total_reward += reward
        prompt = build_prompt(obs)   # update prompt with new terminal state

        if done:
            break

    eval_logger.end_episode()
    eval_rewards.append(total_reward)
    if total_reward > 2.0:
        success_count += 1

print(f'\nEval Results:')
print(f'  Success rate : {success_count}/10')
print(f'  Mean reward  : {sum(eval_rewards) / len(eval_rewards):+.3f}')
print(f'  Rewards      : {[f"{r:+.2f}" for r in eval_rewards]}')
eval_logger.save_log('outputs/eval_log.json')